# Day 052 — Exercise 3: The AI Endpoint

**What you'll build:** `create_chat_app(model)` — a `POST /chat` endpoint that sends the request message to Ollama and returns the reply as a `ChatResponse`. Model errors become an `HTTPException(503)`.

**Why it matters:** This is the endpoint that makes it an *AI* API. It wires the validated request into `ollama.chat` and returns a typed response. The key discipline is error handling: a web handler must never leak a raw exception — you convert a model failure into a clean 503 so clients get a meaningful HTTP status, not a 500 stack trace.

## Provided: Setup + Models (from Exercise 2)

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient
import ollama


class ChatRequest(BaseModel):
    """Request body for the chat endpoints."""
    message: str = Field(min_length=1, description='User message for the model')
    temperature: float = Field(default=0.7, ge=0.0, le=1.0)


class ChatResponse(BaseModel):
    """Response body returned by the chat endpoints."""
    reply: str
    model: str


class HealthResponse(BaseModel):
    """Response body for the health check."""
    status: str
    model: str

## Your Implementation

In [ ]:
def create_chat_app(model: str = 'llama3.2') -> FastAPI:
    """
    POST /chat — send req.message to Ollama, return a ChatResponse.
    On any model error, raise HTTPException(status_code=503, ...).
    """
    app = FastAPI()

    @app.post('/chat', response_model=ChatResponse)
    def chat(req: ChatRequest):
        # TODO: try:
        #     resp = ollama.chat(model=model,
        #         messages=[{'role': 'user', 'content': req.message}],
        #         options={'temperature': req.temperature})
        #     return ChatResponse(reply=resp['message']['content'].strip(), model=model)
        # except Exception as e:
        #     raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')
        pass

    return app

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: create_chat_app returns a FastAPI app
    try:
        app = create_chat_app()
        assert isinstance(app, FastAPI), f'expected FastAPI, got {type(app).__name__}'
        client = TestClient(app)
        passed += 1; print('✅ Check 1: create_chat_app returns a FastAPI app')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: POST /chat returns 200 with a non-empty reply (Ollama)
    try:
        r = client.post('/chat', json={'message': 'Reply with exactly: pong'})
        assert r.status_code == 200, f'expected 200, got {r.status_code} ({r.text[:120]})'
        reply = r.json()['reply']
        assert isinstance(reply, str) and len(reply) > 0, f'empty reply: {r.json()}'
        passed += 1; print(f'✅ Check 2: POST /chat -> 200 with reply ({len(reply)} chars)')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: response reports the model name
    try:
        r = client.post('/chat', json={'message': 'hi'})
        assert r.json().get('model') == 'llama3.2', f"expected model llama3.2, got {r.json()}"
        passed += 1; print('✅ Check 3: response reports model == llama3.2')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: invalid body (missing message) -> 422
    try:
        r = client.post('/chat', json={'temperature': 0.5})
        assert r.status_code == 422, f'expected 422, got {r.status_code}'
        passed += 1; print('✅ Check 4: missing message -> 422')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: a bad model name is turned into a 503, not a crash
    try:
        bad = TestClient(create_chat_app(model='no-such-model-xyz'))
        r = bad.post('/chat', json={'message': 'hi'})
        assert r.status_code == 503, f'expected 503 on bad model, got {r.status_code}'
        passed += 1; print('✅ Check 5: model error -> HTTPException(503)')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def create_chat_app(model: str = 'llama3.2') -> FastAPI:
    """POST /chat — send the message to Ollama and return a ChatResponse.

    Wrap the model call in try/except and convert any failure into an
    HTTPException(503) so the client gets a clean error, not a 500 stack trace.
    """
    app = FastAPI()

    @app.post('/chat', response_model=ChatResponse)
    def chat(req: ChatRequest):
        try:
            resp = ollama.chat(
                model=model,
                messages=[{'role': 'user', 'content': req.message}],
                options={'temperature': req.temperature},
            )
            return ChatResponse(reply=resp['message']['content'].strip(), model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    return app
```

**Why this works:** By the time `chat` runs, `req` is already a validated `ChatRequest` — no manual parsing. The `ollama.chat` call is wrapped so any failure (Ollama down, unknown model) is re-raised as `HTTPException(503)`, which FastAPI turns into a clean JSON error response with that status code. Raising `HTTPException` is how you return non-200 statuses from a handler — the equivalent of the try/except fallback you wrote for the Streamlit app, but expressed in HTTP terms.
</details>